In [ ]:
#  Install Packages
!pip install -q pandas numpy sentence-transformers faiss-cpu transformers accelerate

In [ ]:
!pip install -q rouge-score scikit-learn

In [ ]:
# SECTION 1 - Imports
import re
import torch
import faiss
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# SECTION 2 - Load Data
from google.colab import drive

drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_csv(r"/content/drive/MyDrive/compliance/compliance_dataset_adv.csv")

In [ ]:
df.head()

,product_name,description,category,image_url,brand_name,sku,barcode,price,currency,country_of_origin,...,storage_instructions,expiry_date,material_composition,contains_hazardous_substances,compliance_certifications,manufacturer_details,importer_details,regulatory_body,compliance_status,batch_or_lot_number
0,Surf Excel Easy Wash Detergent 1kg,Surf Excel Easy Wash Detergent 1kg designed fo...,Household,https://example.com/products/hul/1.jpg,HUL,HUL-3748,89844414619507,2101,INR,India,...,"Store in a cool, dry place",NaN,Mixed materials,Yes,FDA,"HUL India Pvt Ltd, Mumbai","ABC Imports Pvt Ltd, Chennai",BIS,Pending,HU65181
1,Tata Salt Iodized 1kg,Tata Salt Iodized 1kg designed for everyday us...,Food,https://example.com/products/tata/2.jpg,Tata,TAT-1868,89506121585909,5437,INR,India,...,Keep away from children,2026-05-07,Plastic and metal components,No,ISI,"Tata India Pvt Ltd, Mumbai","XYZ Traders, Delhi",FSSAI,Compliant,TA48072
2,Skybags Casual Backpack,Skybags Casual Backpack designed for everyday ...,Bags,https://example.com/products/skybags/3.jpg,Skybags,SKY-1151,89328711253397,5858,INR,India,...,Keep away from children,2026-02-11,Plastic and metal components,Yes,NaN,"Skybags India Pvt Ltd, Mumbai","XYZ Traders, Delhi",BIS,Compliant,SK97556
3,Dabur Honey 500g,Dabur Honey 500g designed for everyday use wit...,Food,https://example.com/products/dabur/4.jpg,Dabur,DAB-5779,89496277767469,5069,INR,India,...,Refrigerate after opening,2026-09-12,Mixed materials,Yes,FSSAI,"Dabur India Pvt Ltd, Mumbai","ABC Imports Pvt Ltd, Chennai",CDSCO,Compliant,DA66568
4,Skybags Casual Backpack,Skybags Casual Backpack designed for everyday ...,Bags,https://example.com/products/skybags/5.jpg,Skybags,SKY-3770,89659172180634,4896,INR,India,...,Keep away from children,2025-08-20,Plastic and metal components,No,FSSAI,"Skybags India Pvt Ltd, Mumbai",NaN,BIS,Compliant,SK33976


In [ ]:
# SECTION 2.1 - Quick EDA

dataset_overview = pd.DataFrame({
    "metric": ["rows", "columns", "duplicate_rows", "total_missing_values"],
    "value": [df.shape[0], df.shape[1], df.duplicated().sum(), df.isnull().sum().sum()]
})

column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_values": df.isnull().sum().values,
    "missing_percent": (df.isnull().mean().values * 100).round(2),
    "unique_values": df.nunique().values
})

compliance_distribution = df["compliance_status"].value_counts(dropna=False).reset_index(name="count")
compliance_distribution.columns = ["compliance_status", "count"]
compliance_distribution["percentage"] = (compliance_distribution["count"] / len(df) * 100).round(2)

category_distribution = df["category"].value_counts(dropna=False).reset_index(name="count")
category_distribution.columns = ["category", "count"]
category_distribution["percentage"] = (category_distribution["count"] / len(df) * 100).round(2)

dataset_overview

,metric,value
0,rows,5000
1,columns,25
2,duplicate_rows,0
3,total_missing_values,8546


In [ ]:
# SECTION 2.2 - Preprocessing and Compliance Field Filling

text_columns = df.select_dtypes(include="object").columns
df[text_columns] = df[text_columns].fillna("").astype(str).apply(lambda col: col.str.strip())

numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns
df[numeric_columns] = df[numeric_columns].fillna(0)
df["price"] = pd.to_numeric(df["price"], errors="coerce").fillna(0)

important_columns = [
    "hazard_type", "has_safety_warning", "safety_warning_text",
    "compliance_certifications", "regulatory_body", "compliance_status"
]

for col in important_columns:
    df[col] = (
        df[col].astype(str).str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan, "none": np.nan, "NA": np.nan, "N/A": np.nan})
    )


def fill_with_group_mode(data, column, group_columns, fallback="Undefined"):
    for group_col in group_columns:
        mode_map = (
            data.dropna(subset=[column])
            .groupby(group_col)[column]
            .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
        )
        data[column] = data[column].fillna(data[group_col].map(mode_map))
    data[column] = data[column].fillna(fallback)
    return data


df = fill_with_group_mode(df, "regulatory_body", ["category", "brand_name"], fallback="Undefined")
df = fill_with_group_mode(df, "compliance_certifications", ["category", "brand_name"], fallback="Undefined")


def infer_hazard_type(row):
    product = str(row["product_name"]).lower()
    category = str(row["category"]).lower()
    current = row["hazard_type"]

    no_hazard_keywords = [
        "salt", "honey", "biscuit", "cookies", "tea", "milk", "chocolate",
        "dal", "turmeric", "chilli", "noodles", "glucose", "horlicks", "ghee", "butter"
    ]
    electrical_keywords = [
        "charger", "adapter", "cable", "earbuds", "smartwatch", "monitor",
        "power bank", "kettle", "iron", "heater", "fan", "mixer", "toaster", "battery"
    ]
    chemical_keywords = [
        "detergent", "cleaner", "spray", "shampoo", "lotion", "gel", "toothpaste",
        "sanitizer", "freshener", "cream", "wash", "oil", "soap"
    ]
    physical_keywords = [
        "bag", "backpack", "trolley", "shoes", "sandals", "diaper", "notebook", "marker", "bottle"
    ]

    if any(word in product for word in no_hazard_keywords):
        return "Not Provided"
    if any(word in product for word in electrical_keywords):
        return "Electrical"
    if any(word in product for word in chemical_keywords):
        return "Chemical"
    if any(word in product for word in physical_keywords):
        return "Physical"
    if pd.notna(current):
        return current
    if category in ["electronics", "appliances"]:
        return "Electrical"
    if category in ["beauty", "household", "health"]:
        return "Chemical"
    if category in ["baby care", "bags", "footwear", "stationery"]:
        return "Physical"
    if category == "food":
        return "Not Provided"
    return np.nan


df["hazard_type"] = df.apply(infer_hazard_type, axis=1)
df = fill_with_group_mode(df, "hazard_type", ["category", "brand_name"], fallback="Undefined")


def infer_warning_availability(row):
    current = row["has_safety_warning"]
    warning_text = row["safety_warning_text"]
    hazard = str(row["hazard_type"]).lower().strip()

    if hazard in ["none", "undefined"]:
        return "No"
    if pd.notna(current):
        return current
    if pd.notna(warning_text) and str(warning_text).strip() != "":
        return "Yes"
    return "No"


df["has_safety_warning"] = df.apply(infer_warning_availability, axis=1)


def fill_safety_warning_text(row):
    warning_text = row["safety_warning_text"]
    has_warning = str(row["has_safety_warning"]).strip().lower()
    hazard = str(row["hazard_type"]).strip()

    if pd.notna(warning_text) and str(warning_text).strip() != "":
        return warning_text
    if has_warning == "yes" and hazard.lower() not in ["none", "undefined"]:
        return f"{hazard} risk. Follow safety instructions carefully."
    return "Not Provided"


df["safety_warning_text"] = df.apply(fill_safety_warning_text, axis=1)
df["compliance_status"] = df["compliance_status"].fillna("Needs Review").replace({"Undefined": "Needs Review", "": "Needs Review", "nan": "Needs Review"})

for col in important_columns:
    df[col] = df[col].astype(str).str.strip()

preprocessing_check = pd.DataFrame({
    "column": important_columns,
    "missing_values": [df[col].isna().sum() for col in important_columns],
    "empty_strings": [(df[col] == "").sum() for col in important_columns],
    "unique_values": [df[col].nunique() for col in important_columns]
})

preprocessing_check

,column,missing_values,empty_strings,unique_values
0,hazard_type,0,0,4
1,has_safety_warning,0,0,2
2,safety_warning_text,0,0,4
3,compliance_certifications,0,0,18
4,regulatory_body,0,0,3
5,compliance_status,0,0,3


In [ ]:
# SECTION 2.3 - Create RAG Documents

def create_rag_document(row):
    return f"""
Product Name: {row['product_name']}
Brand: {row['brand_name']}
Category: {row['category']}
Description: {row['description']}
SKU: {row['sku']}
Barcode: {row['barcode']}
Country of Origin: {row['country_of_origin']}
Price: {row['price']} {row['currency']}
Hazard Type: {row['hazard_type']}
Safety Warning Available: {row['has_safety_warning']}
Safety Warning Text: {row['safety_warning_text']}
Age Restriction: {row['age_restriction']}
Usage Instructions: {row['usage_instructions']}
Storage Instructions: {row['storage_instructions']}
Expiry Date: {row['expiry_date']}
Material Composition: {row['material_composition']}
Contains Hazardous Substances: {row['contains_hazardous_substances']}
Compliance Certifications: {row['compliance_certifications']}
Regulatory Body: {row['regulatory_body']}
Manufacturer Details: {row['manufacturer_details']}
Importer Details: {row['importer_details']}
Compliance Status: {row['compliance_status']}
Batch or Lot Number: {row['batch_or_lot_number']}
""".strip()


df["rag_document"] = df.apply(create_rag_document, axis=1)

final_check = pd.DataFrame({
    "check": ["rows", "columns", "empty_rag_documents", "duplicate_skus", "duplicate_barcodes", "duplicate_batch_or_lot_numbers"],
    "value": [
        df.shape[0], df.shape[1], (df["rag_document"].str.len() == 0).sum(),
        df["sku"].duplicated().sum(), df["barcode"].duplicated().sum(),
        df["batch_or_lot_number"].duplicated().sum()
    ]
})

final_check

,check,value
0,rows,5000
1,columns,26
2,empty_rag_documents,0
3,duplicate_skus,0
4,duplicate_barcodes,0
5,duplicate_batch_or_lot_numbers,0


In [ ]:
# SECTION 3 - Rule-Based Signals

def parse_expiry_date(value):
    value = str(value).strip()
    if value.upper() in ["NA", "N/A", "NONE", "", "NAN", "NOT PROVIDED", "UNDEFINED"]:
        return pd.NaT
    return pd.to_datetime(value, errors="coerce")


def rule_based_signal(product):
    issues = []
    category = str(product.get("category", "")).lower().strip()
    hazard = str(product.get("hazard_type", "")).lower().strip()
    warning = str(product.get("has_safety_warning", "")).lower().strip()
    warning_text = str(product.get("safety_warning_text", "")).lower().strip()
    cert = str(product.get("compliance_certifications", "")).lower().strip()
    status = str(product.get("compliance_status", "")).lower().strip()
    expiry = parse_expiry_date(product.get("expiry_date", ""))

    if category in ["food", "beauty", "health", "baby care"] and pd.notna(expiry) and expiry < pd.Timestamp.today():
        issues.append("Expired product")
    if hazard in ["chemical", "electrical"] and (warning != "yes" or warning_text in ["", "not provided", "undefined"]):
        issues.append("Missing safety warning")
    if cert in ["", "not provided", "undefined", "missing or expired certification", "certification under review"]:
        issues.append("Certification issue")
    if status == "non-compliant":
        issues.append("Marked non-compliant")

    risk = "Low" if len(issues) == 0 else "Medium" if len(issues) == 1 else "High"
    return {"rule_risk_level": risk, "rule_issues": issues, "needs_llm": True}


rule_outputs = df.apply(lambda row: rule_based_signal(row.to_dict()), axis=1)
df["rule_risk_level"] = rule_outputs.apply(lambda x: x["rule_risk_level"])
df["rule_issues"] = rule_outputs.apply(lambda x: "; ".join(x["rule_issues"]))
df["needs_llm"] = True

rule_summary = df["rule_risk_level"].value_counts().reset_index(name="count")
rule_summary.columns = ["rule_risk_level", "count"]
rule_summary["percentage"] = (rule_summary["count"] / len(df) * 100).round(2)
rule_summary

,rule_risk_level,count,percentage
0,Low,3471,69.42
1,Medium,1180,23.60
2,High,349,6.98


In [ ]:
# SECTION 4 - Embeddings

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
documents = df["rag_document"].tolist()

embeddings = embedding_model.encode(
    documents,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

(5000, 384)

In [ ]:
# SECTION 5 - FAISS Index

embedding_dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(embeddings)

faiss_index.ntotal

5000

In [ ]:
# SECTION 6 - Retrieval Function

def retrieve_similar_records(query, top_k=5):
    query_embedding = embedding_model.encode(
        [str(query)],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    similarity_scores, retrieved_indices = faiss_index.search(query_embedding, top_k)
    results = df.iloc[retrieved_indices[0]].copy()
    results["similarity_score"] = similarity_scores[0]

    return results[[
        "product_name", "brand_name", "category", "hazard_type", "has_safety_warning",
        "safety_warning_text", "compliance_certifications", "regulatory_body",
        "compliance_status", "rule_risk_level", "rule_issues", "rag_document", "similarity_score"
    ]]

retrieve_similar_records("Horlicks Classic Malt 500g", top_k=3)[[
    "product_name", "category", "hazard_type", "has_safety_warning",
    "compliance_certifications", "compliance_status", "similarity_score"
]]

,product_name,category,hazard_type,has_safety_warning,compliance_certifications,compliance_status,similarity_score
20,Horlicks Classic Malt 500g,Health,Not Provided,Yes,BIS,Compliant,0.602887
4142,Dabur Glucose-D 500g - Combo Pack,Food,Not Provided,Yes,Ayush,Compliant,0.357602
4670,Dabur Glucose-D 500g - Combo Pack,Food,Not Provided,No,Ayush,Compliant,0.351143


In [ ]:
import pickle, faiss
'''df.to_csv("Data.csv", index=False)
with open("embeddings.pkl", "wb") as f:
    pickle.dump(embeddings, f)
faiss.write_index(faiss_index, "faiss_index.bin")
with open("model_config.pkl", "wb") as f:
    pickle.dump({"embedding_model_name": "all-MiniLM-L6-v2",
                 "llm_model_name": "Qwen/Qwen2.5-0.5B-Instruct"}, f)'''

'df.to_csv("Data.csv", index=False)\nwith open("embeddings.pkl", "wb") as f:\n    pickle.dump(embeddings, f)\nfaiss.write_index(faiss_index, "faiss_index.bin")\nwith open("model_config.pkl", "wb") as f:\n    pickle.dump({"embedding_model_name": "all-MiniLM-L6-v2",\n                 "llm_model_name": "Qwen/Qwen2.5-0.5B-Instruct"}, f)'

In [ ]:
def evaluate_retrieval_simple(query, expected, top_k=5):
    results = retrieve_similar_records(query, top_k=top_k)
    retrieved = results["product_name"].str.lower().tolist()

    hit = any(exp.lower() in name for name in retrieved for exp in expected)

    return {
        "query": query,
        "hit_rate_at_k": int(hit)
    }

In [ ]:
test_cases = [
    {"query": "tata salt", "expected": ["tata salt"]},
    {"query": "baby diaper", "expected": ["diaper"]}
]

results = [evaluate_retrieval_simple(tc["query"], tc["expected"]) for tc in test_cases]
print(results)


[{'query': 'tata salt', 'hit_rate_at_k': 1}, {'query': 'baby diaper', 'hit_rate_at_k': 1}]


In [ ]:
# SECTION 7 - LLM Loading

llm_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)


def generate_llm_response(prompt, max_new_tokens=180):
    messages = [
        {"role": "system", "content": "You are a product compliance assistant. Keep answers short and structured."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(llm_model.device)

    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
# SECTION 8 - RAG Compliance System

def analyze_retrieved_evidence(retrieved_df, similarity_threshold=0.35, relative_margin=0.08):
    top_similarity = float(retrieved_df["similarity_score"].iloc[0])

    if top_similarity < similarity_threshold:
        return {
            "decision": "Needs Review",
            "reason": "The product was not found clearly in the dataset.",
            "suggestion": "Enter a more specific product name or manually verify certification, hazard type, and regulatory body."
        }

    relevant_df = retrieved_df[
        retrieved_df["similarity_score"] >= max(similarity_threshold, top_similarity - relative_margin)
    ].copy()

    relevant_df["status_clean"] = relevant_df["compliance_status"].astype(str).str.strip().str.lower()
    relevant_df["hazard_clean"] = relevant_df["hazard_type"].astype(str).str.strip().str.lower()
    relevant_df["warning_clean"] = relevant_df["has_safety_warning"].astype(str).str.strip().str.lower()
    relevant_df["cert_clean"] = relevant_df["compliance_certifications"].astype(str).str.strip().str.lower()

    compliant_count = (relevant_df["status_clean"] == "compliant").sum()
    non_compliant_count = (relevant_df["status_clean"] == "non-compliant").sum()
    pending_count = (relevant_df["status_clean"] == "pending").sum()

    serious_warning_issue_count = (
        relevant_df["hazard_clean"].isin(["chemical", "electrical"]) &
        (relevant_df["warning_clean"] != "yes")
    ).sum()

    missing_cert_count = relevant_df["cert_clean"].isin([
        "", "nan", "none", "not provided", "undefined",
        "missing or expired certification", "certification under review"
    ]).sum()

    if non_compliant_count >= 2:
        return {
            "decision": "Non-Compliant",
            "reason": "Multiple relevant records are marked non-compliant.",
            "suggestion": "Review certification, safety warning, and regulatory approval before sale."
        }
    if serious_warning_issue_count >= 2:
        return {
            "decision": "Non-Compliant",
            "reason": "Relevant records show chemical or electrical hazards without required safety warnings.",
            "suggestion": "Add proper safety warnings and verify label compliance."
        }
    if non_compliant_count == 1 or pending_count > 0:
        return {
            "decision": "Needs Review",
            "reason": "Relevant records contain mixed or pending compliance evidence.",
            "suggestion": "Verify the exact product batch, certification, and regulatory approval before final approval."
        }
    if compliant_count >= 1 and missing_cert_count == 0 and serious_warning_issue_count == 0:
        return {
            "decision": "Compliant",
            "reason": "The relevant retrieved record is compliant with valid certification and required warning evidence.",
            "suggestion": "Maintain updated certification and regulatory documentation."
        }

    return {
        "decision": "Needs Review",
        "reason": "Retrieved evidence is not strong enough for a confident compliance decision.",
        "suggestion": "Verify the exact product certification, hazard type, and regulatory body before approval."
    }

In [ ]:
def _short_text(text, fallback, max_words=28):
    text = str(text).strip()
    if text == "":
        return fallback
    text = re.split(r"(?<=[.!?])\s+", text)[0].strip()
    words = text.split()
    if len(words) > max_words:
        text = " ".join(words[:max_words]).rstrip(",.;") + "."
    return text

In [ ]:

def polish_evidence_with_llm(product_query, evidence):
    prompt = f"""
The compliance decision is already fixed. Do not change it.

Product query: {product_query}
Compliance Decision: {evidence['decision']}
Evidence Reason: {evidence['reason']}
Evidence Suggestion: {evidence['suggestion']}

Rewrite only the reason and suggestion in simple, professional language.
Return exactly two lines:
Reason: one short sentence
Suggestion: one short sentence
""".strip()

    try:
        raw_output = generate_llm_response(prompt, max_new_tokens=120)
        reason = ""
        suggestion = ""

        for line in raw_output.splitlines():
            clean = line.strip()
            lower = clean.lower()
            if lower.startswith("reason:"):
                reason = clean.split(":", 1)[1].strip()
            elif lower.startswith("suggestion:"):
                suggestion = clean.split(":", 1)[1].strip()

        evidence = evidence.copy()
        evidence["reason"] = _short_text(reason, evidence["reason"])
        evidence["suggestion"] = _short_text(suggestion, evidence["suggestion"])
        return evidence
    except Exception:
        return evidence

In [ ]:

# SECTION 8.1 — LLM GENERATION QUALITY METRICS

import re
from rouge_score import rouge_scorer as rouge_lib

def evaluate_llm_generation(product_query, evidence_before_llm, evidence_after_llm):
    """
    Measures quality of LLM's rewritten reason & suggestion.

    Metrics:
        Decision Preservation — Did LLM keep Compliant/Non-Compliant/Needs Review unchanged?
        ROUGE-L               — How much original meaning survived the rewrite?
        Length Ratio          — Is the output concise (not hallucinating extra text)?
        Concise Output        — Did LLM stay within ~1 sentence per field?

    Args:
        evidence_before_llm : dict from analyze_retrieved_evidence()   (rule-based)
        evidence_after_llm  : dict from polish_evidence_with_llm()     (LLM polished)
    """
    scorer = rouge_lib.RougeScorer(["rougeL"], use_stemmer=True)

    # 1. Decision must NEVER change — this is the most critical check
    decision_preserved = evidence_before_llm["decision"] == evidence_after_llm["decision"]

    # 2. ROUGE-L: how faithful is the rewrite to original reason
    reason_rouge = scorer.score(
        evidence_before_llm["reason"],
        evidence_after_llm["reason"]
    )["rougeL"].fmeasure

    # 3. ROUGE-L on suggestion
    suggestion_rouge = scorer.score(
        evidence_before_llm["suggestion"],
        evidence_after_llm["suggestion"]
    )["rougeL"].fmeasure

    # 4. Length ratio — > 1.5x is likely hallucination
    orig_len = len(evidence_before_llm["reason"]) + len(evidence_before_llm["suggestion"])
    new_len  = len(evidence_after_llm["reason"])  + len(evidence_after_llm["suggestion"])
    length_ratio = new_len / orig_len if orig_len > 0 else 1.0

    # 5. Sentence count — should be 1-2 sentences per field max
    reason_sentences     = len(re.findall(r"[.!?]+", evidence_after_llm["reason"]))
    suggestion_sentences = len(re.findall(r"[.!?]+", evidence_after_llm["suggestion"]))
    concise = reason_sentences <= 2 and suggestion_sentences <= 2

    return {
        "query": product_query,
        "decision_preserved": decision_preserved,
        "reason_rouge_l": round(reason_rouge, 4),
        "suggestion_rouge_l": round(suggestion_rouge, 4),
        "avg_rouge_l": round((reason_rouge + suggestion_rouge) / 2, 4),
        "length_ratio": round(length_ratio, 4),
        "length_ok": length_ratio <= 1.5,
        "concise_output": concise
    }


# ── Test Cases
llm_eval_test_queries = [
    "Horlicks Classic Malt 500g",
    "smartphone charger",
    "tata salt",
    "baby diaper",
]

llm_eval_results = []
for query in llm_eval_test_queries:
    retrieved       = retrieve_similar_records(query, top_k=5)
    evidence_rule   = analyze_retrieved_evidence(retrieved)         # before LLM
    evidence_llm    = polish_evidence_with_llm(query, evidence_rule) # after LLM

    metrics = evaluate_llm_generation(query, evidence_rule, evidence_llm)
    llm_eval_results.append(metrics)

llm_eval_df = pd.DataFrame(llm_eval_results)

print("=" * 60)
print("SECTION 8.1 — LLM GENERATION QUALITY METRICS")
print("=" * 60)
print(f"  Decision Preserved : {llm_eval_df['decision_preserved'].mean() * 100:.1f}%   (must be 100%)")
print(f"  Average ROUGE-L    : {llm_eval_df['avg_rouge_l'].mean():.4f}   (target > 0.40)")
print(f"  Length OK Rate     : {llm_eval_df['length_ok'].mean() * 100:.1f}%   (target 100%)")
print(f"  Concise Output     : {llm_eval_df['concise_output'].mean() * 100:.1f}%   (target 100%)")
print()
print(llm_eval_df[[
    "query", "decision_preserved", "avg_rouge_l",
    "length_ratio", "length_ok", "concise_output"
]].to_string(index=False))

SECTION 8.1 — LLM GENERATION QUALITY METRICS
  Decision Preserved : 100.0%   (must be 100%)
  Average ROUGE-L    : 0.4607   (target > 0.40)
  Length OK Rate     : 100.0%   (target 100%)
  Concise Output     : 100.0%   (target 100%)

                     query  decision_preserved  avg_rouge_l  length_ratio  length_ok  concise_output
Horlicks Classic Malt 500g                True       0.0952        0.7215       True            True
        smartphone charger                True       1.0000        1.0000       True            True
                 tata salt                True       0.4000        1.2581       True            True
               baby diaper                True       0.3478        0.8101       True            True


In [ ]:
def format_compliance_output(evidence):
    return f"""Compliance Decision: {evidence['decision']}
Reason: {evidence['reason']}
Suggestion: {evidence['suggestion']}"""


def rag_compliance_check(user_product_query, top_k=5, use_llm=True):
    retrieved_df = retrieve_similar_records(user_product_query, top_k=top_k)
    evidence = analyze_retrieved_evidence(retrieved_df)

    if use_llm:
        evidence = polish_evidence_with_llm(user_product_query, evidence)

    return {
        "user_product_query": user_product_query,
        "top_similarity_score": float(retrieved_df["similarity_score"].iloc[0]),
        "retrieved_records": retrieved_df,
        "llm_output": format_compliance_output(evidence)
    }

In [ ]:
def show_rag_result(result, show_records=False):
    print(result["llm_output"])

    if show_records:
        display(result["retrieved_records"][[
            "product_name", "category", "hazard_type", "has_safety_warning",
            "compliance_certifications", "regulatory_body", "compliance_status", "similarity_score"
        ]])

In [ ]:
# SECTION 9 - Final Wrapper

def check_product_compliance(product_name, use_llm=True):
    product_name = str(product_name).strip()

    if product_name == "":
        return """Compliance Decision: Needs Review
Reason: No product name was entered.
Suggestion: Enter a valid product name for compliance checking."""

    result = rag_compliance_check(product_name, top_k=5, use_llm=use_llm)
    return result["llm_output"]


def check_product_compliance_with_evidence(product_name, use_llm=True):
    product_name = str(product_name).strip()

    if product_name == "":
        return {"final_output": check_product_compliance(product_name), "retrieved_records": None}

    result = rag_compliance_check(product_name, top_k=5, use_llm=use_llm)
    return {"final_output": result["llm_output"], "retrieved_records": result["retrieved_records"]}

product_name = input("Enter product name: ")
print(check_product_compliance(product_name))

Enter product name: horlicks
Compliance Decision: Compliant
Reason: The relevant retrieved record complies with valid certification and required warning evidence.
Suggestion: Ensure ongoing compliance with relevant certifications and regulations.


In [ ]:
# SECTION 10 - Save Artifacts for Streamlit App

import pickle
import faiss
from pathlib import Path

# Create one folder to store everything needed by the Streamlit app
artifact_dir = Path("/content/drive/MyDrive/compliance/app_artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

# 1. Save processed dataset
df.to_csv(artifact_dir / "Data.csv", index=False)

# 2. Save embeddings
with open(artifact_dir / "embeddings.pkl", "wb") as f:
    pickle.dump(embeddings, f)

# 3. Save FAISS index
faiss.write_index(faiss_index, str(artifact_dir / "faiss_index.bin"))

# 4. Save embedding model locally
embedding_model.save(str(artifact_dir / "embedding_model"))

# 5. Save tokenizer and LLM locally
llm_dir = artifact_dir / "llm_qwen"
llm_dir.mkdir(parents=True, exist_ok=True)

tokenizer.save_pretrained(str(llm_dir))
llm_model.save_pretrained(str(llm_dir))

# 6. Save model configuration for the Streamlit app
model_config = {
    "embedding_model_name": "all-MiniLM-L6-v2",
    "llm_model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "embedding_model_path": str(artifact_dir / "embedding_model"),
    "llm_model_path": str(llm_dir),
}

with open(artifact_dir / "model_config.pkl", "wb") as f:
    pickle.dump(model_config, f)

print("All artifacts saved successfully for Streamlit app use.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

All artifacts saved successfully for Streamlit app use.


In [ ]:
# Testing

test_products = [
    "smartphone charger",
    "Horlicks Classic Malt 500g",
    "tata salt",
    "diaper",
    "Puma Sports Sandals - Retail Pack",
    "apple",
    ""
]

for product in test_products:
    print("=" * 80)
    print("Product:", product if product else "EMPTY INPUT")
    print(check_product_compliance(product))

Product: smartphone charger
Compliance Decision: Non-Compliant
Reason: Relevant records show chemical or electrical hazards without required safety warnings.
Suggestion: Add proper safety warnings and verify label compliance.
Product: Horlicks Classic Malt 500g
Compliance Decision: Compliant
Reason: The product meets all applicable regulations and certifications.
Suggestion: Ensure ongoing compliance with relevant standards.
Product: tata salt
Compliance Decision: Needs Review
Reason: The product details are unclear or incomplete.
Suggestion: Verify all relevant information including product batch number, certification status, and regulatory approvals before proceeding with final approval.
Product: diaper
Compliance Decision: Needs Review
Reason: The product details are unclear or conflicting.
Suggestion: Verify all relevant information before final approval.
Product: Puma Sports Sandals - Retail Pack
Compliance Decision: Needs Review
Reason: The product details may include mixed or pe

In [ ]:
# SECTION 11 — SIMPLIFIED END-TO-END ACCURACY

import pandas as pd

def evaluate_compliance_accuracy_simple(test_cases):
    results = []

    for tc in test_cases:
        query = tc["query"]
        expected = tc["expected_decision"]

        result = rag_compliance_check(query, top_k=5, use_llm=True)
        output = result["llm_output"]

        # Extract predicted decision
        predicted = "Needs Review"
        for line in output.splitlines():
            if "compliance decision:" in line.lower():
                predicted = line.split(":", 1)[1].strip()
                break

        results.append({
            "query": query if query else "EMPTY INPUT",
            "expected": expected,
            "predicted": predicted,
            "correct": expected == predicted
        })

    df = pd.DataFrame(results)

    accuracy = df["correct"].mean()

    # ── Print Results ──
    print("=" * 60)
    print("SECTION 11 — END-TO-END ACCURACY (SIMPLIFIED)")
    print("=" * 60)
    print(f"Overall Accuracy: {accuracy * 100:.1f}%\n")

    print(df.to_string(index=False))

    return df, accuracy


# ── Test Cases ──
test_cases = [
    {"query": "Horlicks Classic Malt 500g", "expected_decision": "Compliant"},
    {"query": "smartphone charger", "expected_decision": "Needs Review"},
    {"query": "tata salt", "expected_decision": "Compliant"},
    {"query": "baby diaper", "expected_decision": "Compliant"},
    {"query": "Puma Sports Sandals - Retail Pack", "expected_decision": "Compliant"},
    {"query": "", "expected_decision": "Needs Review"},
]

df, acc = evaluate_compliance_accuracy_simple(test_cases)

SECTION 11 — END-TO-END ACCURACY (SIMPLIFIED)
Overall Accuracy: 50.0%

                            query     expected     predicted  correct
       Horlicks Classic Malt 500g    Compliant     Compliant     True
               smartphone charger Needs Review Non-Compliant    False
                        tata salt    Compliant  Needs Review    False
                      baby diaper    Compliant     Compliant     True
Puma Sports Sandals - Retail Pack    Compliant  Needs Review    False
                      EMPTY INPUT Needs Review  Needs Review     True


In [ ]:
from google.colab import files
import shutil

shutil.make_archive(
    "/content/app_artifacts",
    "zip",
    "/content/drive/MyDrive/compliance/app_artifacts"
)

files.download("/content/app_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>